# NegMAS Analysis Notebook

Loads experiment outputs and generates the 5 core plots required for the project report/presentation.

In [ ]:
from pathlib import Pathx
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style='whitegrid')
results_dir = Path('../results/main')
runs = pd.read_csv(results_dir / 'main_runs.csv')
summary = pd.read_csv(results_dir / 'main_summary.csv')
runs.head()

In [ ]:
# Plot 1: Agreement rate by strategy pairing and deadline
agg = (runs.groupby(['profile_id','n_steps','strategy_buyer','strategy_seller'], as_index=False)['agreement'].mean())
agg['pair'] = agg['strategy_buyer'] + ' vs ' + agg['strategy_seller']
g = sns.catplot(data=agg, x='pair', y='agreement', col='profile_id', hue='n_steps', kind='bar', height=4, aspect=1.4)
for ax in g.axes.flat:
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90)
    ax.set_ylabel('Agreement rate')
    ax.set_xlabel('Strategy pairing')
plt.tight_layout()

In [ ]:
# Plot 2: Rounds-to-agreement distribution
rounds_df = runs[runs['agreement'] == 1].copy()
rounds_df['pair'] = rounds_df['strategy_buyer'] + ' vs ' + rounds_df['strategy_seller']
plt.figure(figsize=(14,6))
sns.boxplot(data=rounds_df, x='pair', y='rounds', hue='profile_id')
plt.xticks(rotation=90)
plt.title('Rounds to agreement (agreements only)')
plt.tight_layout()

In [ ]:
# Plot 3: Welfare distributions
welfare_df = runs.dropna(subset=['welfare']).copy()
welfare_df['pair'] = welfare_df['strategy_buyer'] + ' vs ' + welfare_df['strategy_seller']
plt.figure(figsize=(14,6))
sns.violinplot(data=welfare_df, x='pair', y='welfare', hue='profile_id', cut=0)
plt.xticks(rotation=90)
plt.title('Joint welfare distribution by pairing')
plt.tight_layout()

In [ ]:
# Plot 4: Pareto distance comparison
pareto_df = runs.dropna(subset=['pareto_distance']).copy()
pareto_df['pair'] = pareto_df['strategy_buyer'] + ' vs ' + pareto_df['strategy_seller']
plt.figure(figsize=(14,6))
sns.barplot(data=pareto_df, x='pair', y='pareto_distance', hue='profile_id', errorbar='ci')
plt.xticks(rotation=90)
plt.title('Distance to Pareto frontier (lower is better)')
plt.tight_layout()

In [ ]:
# Plot 5: Utility-space trajectory example (single run)
from src.domain import make_mechanism, make_ufuns
from src.negotiators import strategy_registry

registry = strategy_registry()
session = make_mechanism(n_steps=50)
buyer_ufun, seller_ufun = make_ufuns('A', session.outcome_space)
session.add(registry['SmartAspirationNegotiator'](name='buyer'), ufun=buyer_ufun)
session.add(registry['BoulwareTBNegotiator'](name='seller'), ufun=seller_ufun)
session.run()
session.plot(show_reserved=False)